Toronto Transit Collision

Project description

Dataset URL: https://open.toronto.ca/dataset/police-annual-statistical-report-traffic-collisions/

Set notebook task variables

In [0]:
%python
dbutils.widgets.text("catalog", "workspace", "Catalog name")
dbutils.widgets.text("schema", "default", "Schema name")
dbutils.widgets.text("volume", "raw_files", "Volume name")

Create a managed volume if it doesn't exist

In [0]:
%python
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")

Download the dataset and save it to the volume

Optimizations
- ✅ Implement memory-safe streaming by chunks to handle arbitrarily large CSV files
- ⬜ Store MD5 file hash in Delta Lake Table Properties to skip processing on reruns if the dataset hasn't been refreshed

In [0]:
%python
import os
import requests
import hashlib

# This function returns the MD5 hash of a file
def get_file_md5(path):
    if not os.path.exists(path):
        return None
    hasher = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

# Toronto Open Data is stored in a CKAN instance. It's APIs are documented here:
# https://docs.ckan.org/en/latest/api/

# To hit our API, you'll be making requests to:
base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"

# Datasets are called "packages". Each package can contain many "resources"
# To retrieve the metadata for this package and its resources, use the package name in this page's URL:
url = base_url + "/api/3/action/package_show"
params = { "id": "police-annual-statistical-report-traffic-collisions"}
package = requests.get(url, params = params).json()

# The output path in Databricks Volumes to save the dataset files
output_path = f"/Volumes/{catalog}/{schema}/{volume}"

# To get resource data:
for idx, resource in enumerate(package["result"]["resources"]):

    # for datastore_active resources:
    if resource["datastore_active"]:

        # To get all records in CSV format:
        url = base_url + "/datastore/dump/" + resource["id"]
        response = requests.get(url, stream=True)
        response.raise_for_status()

        file = f"{output_path}/{resource["id"]}.csv"
        with open(file, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

Load dataset as a dataframe and ingest it to a Bronze layer table using Auto Loader

In [0]:
%python
df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .option("rescuedDataColumn", "_rescued_data")
    .load("/Volumes/workspace/default/raw_files/4ad77de0-a9e0-4725-95d9-dbce6776d6cf.csv"))

# Filter for any row where Spark had to rescue data due to the malformed quotes
corrupt_rows = df.filter("_rescued_data IS NOT NULL")
assert corrupt_rows.count() == 0

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.traffic_collisions_bronze")

In [0]:
SELECT *
FROM workspace.default.traffic_collisions_bronze
;